# Notebook 01_1 — Create Dataset (Text + Image (RoBERTa + BEiT + SAINT) Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Men's Shoes (Size 8)

---

Pipeline:
1. Load cleaned panel data (train + val splits)
2. Load pre-trained embeddings from Part 5 prediction zips
3. Compute PCA features (5 components)
4. Compute cluster similarity features (5 KMeans clusters)
5. Compute neighbor distances (5 nearest neighbors per product)
6. Compute weighted substitute prices (BLP-style IV for DoubleML)
7. Join all prediction outputs (level + diff, time-independent + lag1)
8. Save final train and val CSVs as zip files

**Output:**
```
data/dataset_txt_only_False_embeddings_True_train.zip
data/dataset_txt_only_False_embeddings_True_val.zip
```

## ① Mount Drive

In [1]:
# Drive mount not needed for local execution
print('✅ Local mode')

✅ Local mode


## ② Set Working Directory

In [2]:
import os, sys
from pathlib import Path

# Detect project root by walking up from CWD to find 'data/' and 'code/' folders
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

# Change to code/ directory so relative imports and paths work
CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files here: {os.listdir(".")}')

Working directory: /home/iankuzuma/claude_code/demand_modeling/men-8-subcat-split-proper-embedding/oxfords/code
Files here: ['utils', 'main_train_keys.csv', 'main_val_keys.csv', 'requirements.txt', '01_1_create_dataset_txt_img.ipynb', '01_2_create_dataset_txt.ipynb', '02_cluster_centroid_products.ipynb', '02_cluster_centroid_products_random20.ipynb', '03_1_predictive_performance_txt_img.ipynb', '03_2_predictive_performance_txt.ipynb', '04_evaluation.ipynb']


## ③ Imports

In [3]:
import datasets
import pandas as pd
import numpy as np

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
    compute_neighbors_and_distances,
    compute_neighbor_weighted_prices_lagged,
)

## ④ Config

In [4]:
txt_only = False
include_embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_True"

## ⑤ Load Panel Data

- `window = 28` — 28-day rolling average window
- `mod = 4` — every 4th time period (~13 non-overlapping 4-week periods)
- `max_periods = 54` — caps number of time periods

In [5]:
window = 28
mod = 4
max_periods = 54

data_files = {
    "train": ["train-00000-of-00001.parquet"],
    "validation": ["validation-00000-of-00001.parquet"],
}

ds_dict = datasets.load_dataset(
    "parquet",
    data_dir="../data/amzn_shoes_monthly_diffs_ffill_fixed_splits",
    data_files=data_files,
)

ds_val   = ds_dict["validation"]
ds_train = ds_dict["train"]

columns = [
    "ASIN", "SALES_RANK", "PRICE", "BUYBOX_PRICE", "text", "date", "window",
    "REVIEW_COUNT", "RATING", "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
    "Lightning Deals: Upcoming Deal", "Buy Box: Is FBA", "subcat_aggregated",
]

df_val = ds_val.select_columns(columns).to_pandas()
df_val = df_val[df_val["window"] == window].dropna()
df_val["date_t"] = df_val["date"].astype("category").cat.codes
df_val = df_val[df_val["date_t"] <= max_periods]
df_val = df_val[df_val["date_t"] % mod == 0]

df_train = ds_train.select_columns(columns).to_pandas()
df_train = df_train[df_train["window"] == window].dropna()
df_train["date_t"] = df_train["date"].astype("category").cat.codes
df_train = df_train[df_train["date_t"] <= max_periods]
df_train = df_train[df_train["date_t"] % mod == 0]

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_train_val = pd.concat([df_train, df_val], axis=0).reset_index(drop=True)

for col in ['index', 'level_0']:
    if col in df_train_val.columns:
        df_train_val = df_train_val.drop(columns=[col])

num_val   = len(df_val)
num_train = len(df_train)
print(f"Number of train samples: {num_train}")
print(f"Number of val samples:   {num_val}")

df_train_val["date"] = pd.to_datetime(df_train_val["date"]).dt.strftime("%Y-%m-%d")

dummy_subcat = pd.get_dummies(df_train_val["subcat_aggregated"]) * 1
dummy_subcat_names = list(dummy_subcat.columns)
dummy_time = pd.get_dummies(df_train_val["date"]) * 1
dummy_time_names = list(dummy_time.columns)

for col in dummy_subcat.columns:
    df_train_val[col] = dummy_subcat[col].values
for col in dummy_time.columns:
    df_train_val[col] = dummy_time[col].values

df_full = df_train_val.copy()
df_full["date"] = pd.to_datetime(df_full["date"])
df_full.set_index(["ASIN", "date"], inplace=True)

assert df_full.index.is_unique, f"ERROR: index not unique!"
print(f"✅ Index is unique")
print(f"df_full shape: {df_full.shape}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Number of train samples: 1820
Number of val samples:   1834
✅ Index is unique
df_full shape: (3654, 29)


## ⑥ Load Embeddings and Predictions

In [6]:
results_256_time_independent, configs_time_independent = load_pred_and_emb(
    embedding_size=256,
    txt_only=txt_only,
    lag_type="time_independent",
    config_path="utils/paths_config.yaml",
    get_lag2_for_diff=False,
)

results_256_lag1, configs_lag1 = load_pred_and_emb(
    embedding_size=256,
    txt_only=txt_only,
    lag_type="lag1",
    config_path="utils/paths_config.yaml",
    get_lag2_for_diff=False,
)

if not txt_only:
    results_256_lag2, configs_lag2 = load_pred_and_emb(
        embedding_size=256,
        txt_only=txt_only,
        lag_type="lag1",
        config_path="utils/paths_config.yaml",
        get_lag2_for_diff=False,
    )

[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (1834, 258)
  - val_embeddings_diff:     (1703, 130)
  - train_embeddings_levl:   (1820, 258)
  - val_predictions_levl:    (1834, 4)
  - val_predictions_diff:    (1703, 4)
  - train_predictions_levl:  (1820, 4)
  - train_predictions_diff:  (1690, 4)


[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (1703, 258)
  - val_embeddings_diff:     (1572, 130)
  - train_embeddings_levl:   (1690, 258)
  - val_predictions_levl:    (1703, 4)
  - val_predictions_diff:    (1572, 4)
  - train_predictions_levl:  (1690, 4)
  - train_predictions_diff:  (1560, 4)


[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (1703, 258)
  - val_embeddings_diff:     (1572, 130)
  - train_embeddings_levl:   (1690, 258)
  - val_predictions_levl:    (1703, 4)
  - val_predictions_diff:    (1572, 4)
  - train_predictions_levl:  (1690, 4)
  - train_predictions_diff:  (1560, 4)


## ⑦ PCA, Clustering and Similarity Features

**Key fix:** `center_and_norm` concatenates train + val embeddings creating duplicates.
We deduplicate before joining to prevent row explosion in df_full.

In [7]:
pred_and_emb = results_256_time_independent
embeddings = center_and_norm(
    pred_and_emb["embeddings"][2], pred_and_emb["embeddings"][0]
)
print(f"Embedding shape before dedup: {embeddings.shape}")

# Deduplicate — center_and_norm creates duplicate (ASIN, date) pairs
embeddings = embeddings[~embeddings.index.duplicated(keep="last")]
print(f"Embedding shape after dedup:  {embeddings.shape}")

# Align dates to datetime
embeddings.index = embeddings.index.set_levels(
    pd.to_datetime(embeddings.index.get_level_values("date").unique()), level="date"
)

_, cluster_centroids = get_cluster(embeddings, n_clusters=5, n_init=200)
pca_results = get_pca(embeddings, n_components=5)
df_pca = pd.DataFrame(
    pca_results, columns=[f"pca_{i}" for i in range(pca_results.shape[1])],
    index=embeddings.index
)
df_sim = get_similarities(embeddings, cluster_centroids)

df_full = df_full.join(df_pca)
df_full = df_full.join(df_sim)

if include_embeddings:
    df_full = df_full.join(embeddings.add_prefix("emb_"))

assert df_full.index.is_unique, "ERROR: index not unique after embeddings join!"
print(f"✅ Index still unique")
print(f"df_full shape: {df_full.shape}")
print(f"NaN in emb_0: {df_full['emb_0'].isna().sum()} / {len(df_full)}")

Embedding shape before dedup: (3654, 256)
Embedding shape after dedup:  (3654, 256)


✅ Index still unique
df_full shape: (3654, 295)
NaN in emb_0: 0 / 3654


## ⑧ Neighbor Distances

In [8]:
df_neighbor_asins_by_date, df_distance_df_by_date = compute_neighbors_and_distances(
    embeddings, df_full, n_neighbors=5
)

df_full = df_full.join(df_neighbor_asins_by_date)
df_full = df_full.join(df_distance_df_by_date)

print(f"df_full shape after joining neighbors: {df_full.shape}")

Combined neighbor ASINs shape: (3393, 5)
Combined neighbor distances shape: (3393, 5)
df_full shape after joining neighbors: (3654, 305)


## ⑨ Preview

In [9]:
df_full.head()

SALES_RANK     PRICE  BUYBOX_PRICE  \
ASIN       date                                             
B0007TQ9OK 2025-03-03  -10.011007  3.818426      3.714965   
           2025-03-31  -10.282166  3.816943      3.693067   
           2025-04-28  -10.128009  3.846082      3.710256   
           2025-05-26  -10.245004  3.911023      3.591148   
           2025-06-23  -10.301829  3.815724      3.766807   

                                                                    text  \
ASIN       date                                                            
B0007TQ9OK 2025-03-03  Dockers Gordon Leather Dress Shoes for Men Cas...   
           2025-03-31  Dockers Gordon Leather Dress Shoes for Men Cas...   
           2025-04-28  Dockers Gordon Leather Dress Shoes for Men Cas...   
           2025-05-26  Dockers Gordon Leather Dress Shoes for Men Cas...   
           2025-06-23  Dockers Gordon Leather Dress Shoes for Men Cas...   

                       window  REVIEW_COUNT  RATING  New Offer Count: Current  \
ASIN       date                                                                 
B0007TQ9OK 2025-03-03    28.0        8631.0     4.4                         3   
           2025-03-31    28.0        8655.0     4.4                         3   
           2025-04-28    28.0        8673.0     4.4                         3   
           2025-05-26    28.0        8698.0     4.4                         3   
           2025-06-23    28.0        8809.0     4.4                         2   

                       Count of retrieved live offers: New, FBA  \
ASIN       date                                                   
B0007TQ9OK 2025-03-03                                         0   
           2025-03-31                                         0   
           2025-04-28                                         0   
           2025-05-26                                         0   
           2025-06-23                                         0   

                       Count of retrieved live offers: New, FBM  ...  \
ASIN       date                                                  ...   
B0007TQ9OK 2025-03-03                                         0  ...   
           2025-03-31                                         0  ...   
           2025-04-28                                         0  ...   
           2025-05-26                                         0  ...   
           2025-06-23                                         0  ...   

                       neighbor_asin_1  neighbor_asin_2 neighbor_asin_3  \
ASIN       date                                                           
B0007TQ9OK 2025-03-03              NaN              NaN             NaN   
           2025-03-31       B01FYE0FZ6       B01FYE0PN8      B07S7P8QZL   
           2025-04-28       B01FYE0FZ6       B01FYE0PN8      B07S7P8QZL   
           2025-05-26       B01FYE0FZ6       B01FYE0PN8      B07S7P8QZL   
           2025-06-23       B01FYE0FZ6       B01FYE0PN8      B07S7P8QZL   

                       neighbor_asin_4  neighbor_asin_5  neighbor_distance_1  \
ASIN       date                                                                
B0007TQ9OK 2025-03-03              NaN              NaN                  NaN   
           2025-03-31       B07SBT7BFX       B01H7S0YAS             0.035579   
           2025-04-28       B07SBT7BFX       B01H7S0YAS             0.035596   
           2025-05-26       B07SBT7BFX       B01H7S0YAS             0.035730   
           2025-06-23       B07SBT7BFX       B01H7S0YAS             0.029242   

                       neighbor_distance_2  neighbor_distance_3  \
ASIN       date                                                   
B0007TQ9OK 2025-03-03                  NaN                  NaN   
           2025-03-31             0.042481             0.044822   
           2025-04-28             0.042365             0.044871   
           2025-05-26             0.042321             0.044732   
           2025-06-23         

## ⑩ Weighted Substitute Prices (Instrument)

Distance-weighted average of 5 nearest neighbors prices lagged one period.
Key demand instrument for DoubleML in notebook 04.

In [10]:
df_full.columns = [str(c) for c in df_full.columns]

weighted_substitute_price = compute_neighbor_weighted_prices_lagged(
    df_full, "BUYBOX_PRICE"
)
df_full = df_full.join(weighted_substitute_price)
df_full.loc[:, ["BUYBOX_PRICE", "weighted_substitute_price_lagged"]].head()

BUYBOX_PRICE  weighted_substitute_price_lagged
ASIN       date                                                      
B0007TQ9OK 2025-03-03      3.714965                               NaN
           2025-03-31      3.693067                          3.652943
           2025-04-28      3.710256                          3.765883
           2025-05-26      3.591148                          3.731595
           2025-06-23      3.766807                          3.729413

## ⑪ Split Back into Train and Val

In [11]:
df_full_val   = df_full.iloc[num_train:, :].copy()
df_full_train = df_full.iloc[:num_train, :].copy()

print(f"df_full shape:       {df_full.shape}")
print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape:   {df_full_val.shape}")

df_full shape:       (3654, 306)
df_full_train shape: (1820, 306)
df_full_val shape:   (1834, 306)


## ⑫ NaN Check

In [12]:
print(df_full_val.isna().sum().sort_values(ascending=False).head(20))

weighted_substitute_price_lagged    131
neighbor_distance_5                 131
neighbor_distance_4                 131
neighbor_distance_3                 131
neighbor_distance_1                 131
neighbor_asin_5                     131
neighbor_asin_4                     131
neighbor_asin_3                     131
neighbor_asin_2                     131
neighbor_asin_1                     131
neighbor_distance_2                 131
emb_156                               0
emb_160                               0
emb_159                               0
emb_158                               0
emb_157                               0
emb_162                               0
emb_155                               0
emb_154                               0
emb_153                               0
dtype: int64


In [13]:
print(df_full_train.isna().sum().sort_values(ascending=False).head(20))

weighted_substitute_price_lagged    130
neighbor_distance_5                 130
neighbor_distance_4                 130
neighbor_distance_3                 130
neighbor_distance_1                 130
neighbor_asin_5                     130
neighbor_asin_4                     130
neighbor_asin_3                     130
neighbor_asin_2                     130
neighbor_asin_1                     130
neighbor_distance_2                 130
emb_156                               0
emb_160                               0
emb_159                               0
emb_158                               0
emb_157                               0
emb_162                               0
emb_155                               0
emb_154                               0
emb_153                               0
dtype: int64


## ⑬ Add Lag Features and Join Model Predictions

**Key fix:** Convert prediction dates to datetime before joining.
This ensures the ASIN+date index matches df_full_train and df_full_val.

In [14]:
df_full_val = df_full_val.rename(
    columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}
)
for i in range(1, 3):
    df_full_val[f"Q_t-{i}"]            = df_full_val.groupby("ASIN")["Q_t"].shift(i)
    df_full_val[f"P_bb_t-{i}"]         = df_full_val.groupby("ASIN")["P_bb_t"].shift(i)
    df_full_val[f"REVIEW_COUNT_t-{i}"] = df_full_val["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_val[f"RATING_t-{i}"]       = df_full_val["RATING"].groupby("ASIN").shift(i)

df_full_val["Delta_Q_t"]    = df_full_val["Q_t"]    - df_full_val["Q_t-1"]
df_full_val["Delta_P_bb_t"] = df_full_val["P_bb_t"] - df_full_val["P_bb_t-1"]

df_full_train = df_full_train.rename(
    columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}
)
for i in range(1, 3):
    df_full_train[f"Q_t-{i}"]            = df_full_train["Q_t"].groupby("ASIN").shift(i)
    df_full_train[f"P_bb_t-{i}"]         = df_full_train["P_bb_t"].groupby("ASIN").shift(i)
    df_full_train[f"REVIEW_COUNT_t-{i}"] = df_full_train["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_train[f"RATING_t-{i}"]       = df_full_train["RATING"].groupby("ASIN").shift(i)

df_full_train["Delta_Q_t"]    = df_full_train["Q_t"]    - df_full_train["Q_t-1"]
df_full_train["Delta_P_bb_t"] = df_full_train["P_bb_t"] - df_full_train["P_bb_t-1"]

# KEY FIX: convert prediction dates to datetime before joining
# This ensures the ASIN+date index matches df_full_train/val datetime index
def prep_pred(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index(["ASIN", "date"])

val_predictions_levl   = prep_pred(pred_and_emb["predictions"][0])
val_predictions_diff   = prep_pred(pred_and_emb["predictions"][1]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"}
)
train_predictions_levl = prep_pred(pred_and_emb["predictions"][2])
train_predictions_diff = prep_pred(pred_and_emb["predictions"][3]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"}
)

df_full_val   = df_full_val.join(val_predictions_levl)
df_full_train = df_full_train.join(train_predictions_levl)
df_full_val   = df_full_val.join(val_predictions_diff)
df_full_train = df_full_train.join(train_predictions_diff)

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape:   {df_full_val.shape}")
print(f"NaN in pred_ml_l train: {df_full_train['pred_ml_l'].isna().sum()} / {len(df_full_train)}")
print(f"NaN in pred_ml_l val:   {df_full_val['pred_ml_l'].isna().sum()} / {len(df_full_val)}")

df_full_train shape: (1820, 320)
df_full_val shape:   (1834, 320)
NaN in pred_ml_l train: 0 / 1820
NaN in pred_ml_l val:   0 / 1834


## ⑭ Join Lag1 Predictions

In [15]:
val_predictions_levl_lag1   = prep_pred(results_256_lag1["predictions"][0]).rename(
    columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"}
)
val_predictions_diff_lag1   = prep_pred(results_256_lag1["predictions"][1]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"}
)
train_predictions_levl_lag1 = prep_pred(results_256_lag1["predictions"][2]).rename(
    columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"}
)
train_predictions_diff_lag1 = prep_pred(results_256_lag1["predictions"][3]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"}
)

df_full_val   = df_full_val.join(val_predictions_levl_lag1)
df_full_train = df_full_train.join(train_predictions_levl_lag1)
df_full_val   = df_full_val.join(val_predictions_diff_lag1)
df_full_train = df_full_train.join(train_predictions_diff_lag1)

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape:   {df_full_val.shape}")

df_full_train shape: (1820, 324)
df_full_val shape:   (1834, 324)


## ⑮ Check Unique Dates

In [16]:
df_full_val.reset_index()["date"].dropna().unique()

<DatetimeArray>
['2025-03-03 00:00:00', '2025-03-31 00:00:00', '2025-04-28 00:00:00',
 '2025-05-26 00:00:00', '2025-06-23 00:00:00', '2025-07-21 00:00:00',
 '2025-08-18 00:00:00', '2025-09-15 00:00:00', '2025-10-13 00:00:00',
 '2025-11-10 00:00:00', '2025-12-08 00:00:00', '2026-01-05 00:00:00',
 '2026-02-02 00:00:00', '2026-03-02 00:00:00']
Length: 14, dtype: datetime64[ns]

## ⑯ Drop Redundant Columns

In [17]:
drop_cols = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
    "pred_ml_l_lag_1", "pred_ml_m_lag_1",
]
keep_cols = [c for c in df_full_val.columns if c not in drop_cols]

## ⑰ Final NaN Check

In [18]:
print(df_full_val[keep_cols].isna().sum().sort_values(ascending=False).head(20))
print("\n\n")
print(df_full_train[keep_cols].isna().sum().sort_values(ascending=False).head(20))

pred_ml_l_diff_lag_1                262
pred_ml_m_diff_lag_1                262
pred_ml_m_diff                      131
pred_ml_l_diff                      131
Delta_P_bb_t                        131
Delta_Q_t                           131
RATING_t-1                          131
REVIEW_COUNT_t-1                    131
P_bb_t-1                            131
Q_t-1                               131
weighted_substitute_price_lagged    131
neighbor_distance_5                 131
neighbor_distance_4                 131
neighbor_distance_3                 131
neighbor_distance_2                 131
neighbor_distance_1                 131
neighbor_asin_5                     131
neighbor_asin_2                     131
neighbor_asin_4                     131
neighbor_asin_3                     131
dtype: int64



pred_ml_l_diff_lag_1                260
pred_ml_m_diff_lag_1                260
pred_ml_m_diff                      130
pred_ml_l_diff                      130
Delta_P_bb_t            

## ⑱ Rename Weighted Substitute Price Column

In [19]:
df_full_val = df_full_val.rename(
    columns={"weighted_substitute_price_lagged": "weighted_substitute_price"}
)
df_full_train = df_full_train.rename(
    columns={"weighted_substitute_price_lagged": "weighted_substitute_price"}
)

## ⑲ Inspect Final Columns

In [20]:
list(df_full_val.columns)

['Q_t',
 'PRICE',
 'P_bb_t',
 'text',
 'window',
 'REVIEW_COUNT',
 'RATING',
 'New Offer Count: Current',
 'Count of retrieved live offers: New, FBA',
 'Count of retrieved live offers: New, FBM',
 'Lightning Deals: Upcoming Deal',
 'Buy Box: Is FBA',
 'subcat_aggregated',
 'date_t',
 'Oxfords',
 '2025-03-03',
 '2025-03-31',
 '2025-04-28',
 '2025-05-26',
 '2025-06-23',
 '2025-07-21',
 '2025-08-18',
 '2025-09-15',
 '2025-10-13',
 '2025-11-10',
 '2025-12-08',
 '2026-01-05',
 '2026-02-02',
 '2026-03-02',
 'pca_0',
 'pca_1',
 'pca_2',
 'pca_3',
 'pca_4',
 'similarity_cluster_0',
 'similarity_cluster_1',
 'similarity_cluster_2',
 'similarity_cluster_3',
 'similarity_cluster_4',
 'emb_0',
 'emb_1',
 'emb_2',
 'emb_3',
 'emb_4',
 'emb_5',
 'emb_6',
 'emb_7',
 'emb_8',
 'emb_9',
 'emb_10',
 'emb_11',
 'emb_12',
 'emb_13',
 'emb_14',
 'emb_15',
 'emb_16',
 'emb_17',
 'emb_18',
 'emb_19',
 'emb_20',
 'emb_21',
 'emb_22',
 'emb_23',
 'emb_24',
 'emb_25',
 'emb_26',
 'emb_27',
 'emb_28',
 'emb_29',

In [21]:
list(df_full_train.columns)

['Q_t',
 'PRICE',
 'P_bb_t',
 'text',
 'window',
 'REVIEW_COUNT',
 'RATING',
 'New Offer Count: Current',
 'Count of retrieved live offers: New, FBA',
 'Count of retrieved live offers: New, FBM',
 'Lightning Deals: Upcoming Deal',
 'Buy Box: Is FBA',
 'subcat_aggregated',
 'date_t',
 'Oxfords',
 '2025-03-03',
 '2025-03-31',
 '2025-04-28',
 '2025-05-26',
 '2025-06-23',
 '2025-07-21',
 '2025-08-18',
 '2025-09-15',
 '2025-10-13',
 '2025-11-10',
 '2025-12-08',
 '2026-01-05',
 '2026-02-02',
 '2026-03-02',
 'pca_0',
 'pca_1',
 'pca_2',
 'pca_3',
 'pca_4',
 'similarity_cluster_0',
 'similarity_cluster_1',
 'similarity_cluster_2',
 'similarity_cluster_3',
 'similarity_cluster_4',
 'emb_0',
 'emb_1',
 'emb_2',
 'emb_3',
 'emb_4',
 'emb_5',
 'emb_6',
 'emb_7',
 'emb_8',
 'emb_9',
 'emb_10',
 'emb_11',
 'emb_12',
 'emb_13',
 'emb_14',
 'emb_15',
 'emb_16',
 'emb_17',
 'emb_18',
 'emb_19',
 'emb_20',
 'emb_21',
 'emb_22',
 'emb_23',
 'emb_24',
 'emb_25',
 'emb_26',
 'emb_27',
 'emb_28',
 'emb_29',

## ⑳ Save Output

**Output files (input to notebooks 03_1 and 04):**
```
data/dataset_txt_only_False_embeddings_True_train.zip
data/dataset_txt_only_False_embeddings_True_val.zip
```

In [22]:
zip_train = dict(method="zip", archive_name=f"../data/{dataframe_name}_train.csv")
df_full_train.to_csv(f"../data/{dataframe_name}_train.zip", compression=zip_train)
zip_val = dict(method="zip", archive_name=f"../data/{dataframe_name}_val.csv")
df_full_val.to_csv(f"../data/{dataframe_name}_val.zip", compression=zip_val)
print(f"✅ Saved: {dataframe_name}_train.zip")
print(f"✅ Saved: {dataframe_name}_val.zip")

✅ Saved: dataset_txt_only_False_embeddings_True_train.zip
✅ Saved: dataset_txt_only_False_embeddings_True_val.zip
